# Delta Analysis: Complexity Features Driving Quantum Advantage

Ridge regression with bootstrapping to identify which complexity features predict quantum vs classical performance differences.

## 1. Setup and Load Delta Analysis Data

In [1]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.linear_model import RidgeCV, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Try to import statsmodels, fall back to custom implementation
try:
    from statsmodels.stats.multitest import multipletests
    USE_STATSMODELS = True
    print("Using statsmodels for FDR correction")
except ImportError:
    USE_STATSMODELS = False
    print("statsmodels not available, using custom FDR implementation")

# Publication-quality settings
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

sns.set_style('whitegrid', {'grid.linestyle': '--', 'grid.alpha': 0.3})

# Load delta analysis CSV
PATH = '../../results/meta_analysis/'
delta_df = pd.read_csv(PATH + 'delta_analysis_best_methods_with_complexity.csv')

print(f"Loaded delta analysis: {delta_df.shape}")
print(f"Columns: {delta_df.columns.tolist()[:20]}...")

Using statsmodels for FDR correction
Loaded delta analysis: (111, 126)
Columns: ['base_dataset', 'task', 'best_quantum_method', 'best_classical_method', 'quantum_median', 'quantum_mean', 'quantum_std', 'classical_median', 'classical_mean', 'classical_std', 'delta_median', 'delta_mean', 'percent_improvement_median', 'percent_improvement_mean', 'p_value', 'significance', 'n_quantum_reps', 'n_classical_reps', 'num_nodes_mean', 'num_nodes_median']...


In [15]:
[x for x in delta_df.columns]

['base_dataset',
 'task',
 'best_quantum_method',
 'best_classical_method',
 'quantum_median',
 'quantum_mean',
 'quantum_std',
 'classical_median',
 'classical_mean',
 'classical_std',
 'delta_median',
 'delta_mean',
 'percent_improvement_median',
 'percent_improvement_mean',
 'p_value',
 'significance',
 'n_quantum_reps',
 'n_classical_reps',
 'num_nodes_mean',
 'num_nodes_median',
 'num_nodes_std',
 'num_edges_mean',
 'num_edges_median',
 'num_edges_std',
 'spectral_gap_mean',
 'spectral_gap_median',
 'spectral_gap_std',
 'algebraic_connectivity_mean',
 'algebraic_connectivity_median',
 'algebraic_connectivity_std',
 'spectral_entropy_mean',
 'spectral_entropy_median',
 'spectral_entropy_std',
 'von_neumann_entropy_mean',
 'von_neumann_entropy_median',
 'von_neumann_entropy_std',
 'quantum_complexity_mean',
 'quantum_complexity_median',
 'quantum_complexity_std',
 'modularity_mean',
 'modularity_median',
 'modularity_std',
 'clustering_mean_mean',
 'clustering_mean_median',
 'cluste

## 2. Data Cleaning - Remove High-NaN Columns

In [7]:
# Calculate NaN percentage for each column
nan_percentages = (delta_df.isna().sum() / len(delta_df)) * 100

# Identify columns with >50% NaNs
high_nan_cols = nan_percentages[nan_percentages > 50].index.tolist()
print(f"\nColumns with >50% NaNs: {len(high_nan_cols)}")
if len(high_nan_cols) > 0:
    print("Dropping:", high_nan_cols[:10], "..." if len(high_nan_cols) > 10 else "")

# Drop high-NaN columns
delta_df_clean = delta_df.drop(columns=high_nan_cols)

# Define specific complexity features to use (using _mean versions)
base_complexity_features = [
    'num_nodes_mean', 'num_edges_mean', 'spectral_gap_mean', 
    'algebraic_connectivity_mean', 'spectral_entropy_mean', 
    'von_neumann_entropy_mean', 'quantum_complexity_mean', 
    'modularity_mean', 'clustering_mean_mean', 'degree_heterogeneity_mean', 
    'quantum_advantage_score_mean', 'cyclomatic_number_mean', 
    'kirchhoff_index_mean', 'qbc_intrinsic_dimension_mean', 
    'qbc_total_correlations_mean', 'qbc_variation_mean',
    'qbc_num_non_zero_entries_mean', 'qbc_num_low_variance_features_mean',
    'qbc_coefficient_of_variation_mean', 'qbc_skewness_mean', 
    'qbc_kurtosis_mean', 'qbc_mean_log_kernel_density_mean', 
    'qbc_isomap_reconstruction_error_mean', 'qbc_fractal_dimension_mean', 
    'qbc_mutual_information_mean'
]

# Filter to only include columns that exist in the dataframe
complexity_cols = [col for col in base_complexity_features if col in delta_df_clean.columns]

print(f"\nAfter cleaning:")
print(f"  Total columns: {delta_df_clean.shape[1]}")
print(f"  Complexity features (using _mean): {len(complexity_cols)}")
print(f"  Rows: {delta_df_clean.shape[0]}")
print(f"\nUsing complexity features:")
for col in complexity_cols:
    print(f"  - {col}")

# Show remaining NaN percentages for complexity features
remaining_nans = (delta_df_clean[complexity_cols].isna().sum() / len(delta_df_clean)) * 100
print(f"\nRemaining NaN percentages in complexity features:")
print(f"  Max: {remaining_nans.max():.1f}%")
print(f"  Mean: {remaining_nans.mean():.1f}%")
print(f"  Features with >20% NaNs: {(remaining_nans > 20).sum()}")



Columns with >50% NaNs: 6
Dropping: ['qbc_fisher_discriminant_ratio_mean', 'qbc_fisher_discriminant_ratio_median', 'qbc_fisher_discriminant_ratio_std', 'qbc_mutual_information_mean', 'qbc_mutual_information_median', 'qbc_mutual_information_std'] 

After cleaning:
  Total columns: 120
  Complexity features (using _mean): 24
  Rows: 111

Using complexity features:
  - num_nodes_mean
  - num_edges_mean
  - spectral_gap_mean
  - algebraic_connectivity_mean
  - spectral_entropy_mean
  - von_neumann_entropy_mean
  - quantum_complexity_mean
  - modularity_mean
  - clustering_mean_mean
  - degree_heterogeneity_mean
  - quantum_advantage_score_mean
  - cyclomatic_number_mean
  - kirchhoff_index_mean
  - qbc_intrinsic_dimension_mean
  - qbc_total_correlations_mean
  - qbc_variation_mean
  - qbc_num_non_zero_entries_mean
  - qbc_num_low_variance_features_mean
  - qbc_coefficient_of_variation_mean
  - qbc_skewness_mean
  - qbc_kurtosis_mean
  - qbc_mean_log_kernel_density_mean
  - qbc_isomap_reco

(111, 120)

## 3. FDR Correction Function

In [9]:
def benjamini_hochberg_fdr(p_values, alpha=0.05):
    """Benjamini-Hochberg FDR correction (custom implementation)"""
    p_values = np.array(p_values)
    n = len(p_values)
    
    sorted_indices = np.argsort(p_values)
    sorted_p = p_values[sorted_indices]
    
    bh_critical = (np.arange(1, n + 1) / n) * alpha
    
    comparisons = sorted_p <= bh_critical
    if comparisons.any():
        max_i = np.where(comparisons)[0].max()
        reject_sorted = np.zeros(n, dtype=bool)
        reject_sorted[:max_i + 1] = True
    else:
        reject_sorted = np.zeros(n, dtype=bool)
    
    reject = np.zeros(n, dtype=bool)
    reject[sorted_indices] = reject_sorted
    
    p_corrected = np.zeros(n)
    p_corrected[sorted_indices] = np.minimum.accumulate(
        sorted_p * n / np.arange(1, n + 1)[::-1][::-1]
    )[::-1][::-1]
    p_corrected = np.minimum(p_corrected, 1.0)
    
    return reject, p_corrected

# Test
test_p = np.array([0.001, 0.01, 0.05, 0.1, 0.5])
reject, p_corr = benjamini_hochberg_fdr(test_p, alpha=0.05)
print("\nFDR correction test:")
print(f"  Original p-values: {test_p}")
print(f"  Corrected p-values: {p_corr}")
print(f"  Reject H0: {reject}")


FDR correction test:
  Original p-values: [0.001 0.01  0.05  0.1   0.5  ]
  Corrected p-values: [0.005 0.005 0.005 0.005 0.005]
  Reject H0: [ True  True False False False]


## 4. Prepare Data for Ridge Regression

In [10]:
def prepare_regression_data(df, task_name, complexity_features):
    """Prepare data for ridge regression for a specific task"""
    task_df = df[df['task'] == task_name].copy()
    
    y = task_df['percent_improvement_mean'].values
    X = task_df[complexity_features].copy()
    
    # Impute remaining NaNs with median
    for col in X.columns:
        if X[col].isna().any():
            X[col].fillna(X[col].median(), inplace=True)
    
    # Remove constant features
    valid_cols = []
    for col in X.columns:
        if X[col].std() > 0 and not X[col].isna().all():
            valid_cols.append(col)
    
    X = X[valid_cols]
    
    print(f"\n{task_name}:")
    print(f"  Samples: {len(y)}")
    print(f"  Features: {len(X.columns)}")
    print(f"  Target range: [{y.min():.2f}, {y.max():.2f}]")
    
    return X, y, valid_cols

# Prepare data for each task
tasks_data = {}
for task in ['Ranking', 'Classification', 'Link Prediction']:
    X, y, features = prepare_regression_data(delta_df_clean, task, complexity_cols)
    tasks_data[task] = {'X': X, 'y': y, 'features': features}


Ranking:
  Samples: 37
  Features: 23
  Target range: [-100.00, 1900.00]

Classification:
  Samples: 37
  Features: 23
  Target range: [-50.03, 20.55]

Link Prediction:
  Samples: 37
  Features: 23
  Target range: [-20.25, 8.82]


In [13]:
delta_df_clean

,base_dataset,task,best_quantum_method,best_classical_method,quantum_median,quantum_mean,quantum_std,classical_median,classical_mean,classical_std,...,qbc_isomap_reconstruction_error_std,qbc_fractal_dimension_mean,qbc_fractal_dimension_median,qbc_fractal_dimension_std,qbc_entropy_mean,qbc_entropy_median,qbc_entropy_std,qbc_std_entropy_mean,qbc_std_entropy_median,qbc_std_entropy_std
0,BioPlex3_asthma,Ranking,quvine_fused,netmf,0.100000,0.100000,0.000000e+00,0.200000,0.200000,0.000000e+00,...,1.841447e-10,1.802163,1.802167,6.009823e-04,1.0,1.0,1.111613e-16,0.0,0.0,0.0
1,BioPlex3_asthma,Classification,quvine_heat,netmf,0.636090,0.636090,0.000000e+00,0.586190,0.586190,0.000000e+00,...,1.841447e-10,1.802163,1.802167,6.009823e-04,1.0,1.0,1.111613e-16,0.0,0.0,0.0
2,BioPlex3_asthma,Link Prediction,quvine_heat,netmf,0.913951,0.913951,0.000000e+00,0.951214,0.951189,5.058902e-05,...,1.841447e-10,1.802163,1.802167,6.009823e-04,1.0,1.0,1.111613e-16,0.0,0.0,0.0
3,BioPlex3_autism,Ranking,quvine_pgcnmf,node2vec,0.000000,0.012500,3.349321e-02,0.000000,0.020000,4.050957e-02,...,9.009572e-10,1.807683,1.807757,6.742326e-04,1.0,1.0,1.111613e-16,0.0,0.0,0.0
4,BioPlex3_autism,Classification,quvine_heat,netmf,0.643545,0.643545,0.000000e+00,0.588106,0.588106,0.000000e+00,...,9.009572e-10,1.807683,1.807757,6.742326e-04,1.0,1.0,1.111613e-16,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,graphs-datasets__twitch_egos_n20000,Classification,quvine_poly,graphsage,0.843097,0.843097,1.129203e-16,0.811920,0.811920,3.387608e-16,...,1.837888e-02,2.009416,2.009060,6.461427e-04,1.0,1.0,1.111768e-16,0.0,0.0,0.0
107,graphs-datasets__twitch_egos_n20000,Link Prediction,quvine_heat,baseline_gcnmf,0.973380,0.973380,0.000000e+00,0.906250,0.907330,1.991588e-03,...,1.837888e-02,2.009416,2.009060,6.461427e-04,1.0,1.0,1.111768e-16,0.0,0.0,0.0
108,graphs-datasets__twitch_egos_n5000,Ranking,quvine_ctqw,node2vec,0.200000,0.200000,5.646013e-17,0.100000,0.130000,8.366600e-02,...,2.223536e-16,2.009060,2.009060,4.447073e-16,1.0,1.0,1.111768e-16,0.0,0.0,0.0
109,graphs-datasets__twitch_egos_n5000,Classification,quvine_poly,graphsage,0.843097,0.843097,1.129203e-16,0.811920,0.811920,3.387608e-16,...,2.223536e-16,2.009060,2.009060,4.447073e-16,1.0,1.0,1.111768e-16,0.0,0.0,0.0


## 5. Ridge Regression with Bootstrapping

In [ ]:
def bootstrap_ridge_regression(X_df, y, feature_names, n_bootstrap=1000, alpha_range=np.logspace(-3, 3, 50)):
    """
    Perform ridge regression with bootstrapping
    Uses grid search to find optimal alpha, then fixes it for bootstrap
    """
    n_samples, n_features = X_df.shape
    X = X_df.values  # Convert to numpy array
    
    # Standardize features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Grid search for optimal alpha using cross-validation
    print(f"  Running grid search over {len(alpha_range)} alpha values...")
    ridge_cv = RidgeCV(alphas=alpha_range, cv=5, scoring='r2')
    ridge_cv.fit(X_scaled, y)
    optimal_alpha = ridge_cv.alpha_
    
    # Get cross-validation score with optimal alpha
    cv_scores = cross_val_score(Ridge(alpha=optimal_alpha), X_scaled, y, cv=5, scoring='r2')
    
    print(f"  Optimal alpha: {optimal_alpha:.4f}")
    print(f"  CV R² (mean ± std): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    
    # Bootstrap with FIXED optimal alpha
    print(f"  Running {n_bootstrap} bootstrap iterations with fixed alpha={optimal_alpha:.4f}...")
    bootstrap_coefs = np.zeros((n_bootstrap, n_features))
    
    for i in range(n_bootstrap):
        indices = np.random.choice(n_samples, size=n_samples, replace=True)
        X_boot = X_scaled[indices]
        y_boot = y[indices]
        
        ridge = Ridge(alpha=optimal_alpha)
        ridge.fit(X_boot, y_boot)
        bootstrap_coefs[i] = ridge.coef_
    
    # Calculate statistics
    coefs_mean = bootstrap_coefs.mean(axis=0)
    coefs_std = bootstrap_coefs.std(axis=0)
    
    # Calculate p-values (two-tailed test)
    p_values = np.zeros(n_features)
    for j in range(n_features):
        if coefs_mean[j] > 0:
            p_values[j] = 2 * (bootstrap_coefs[:, j] < 0).mean()
        else:
            p_values[j] = 2 * (bootstrap_coefs[:, j] > 0).mean()
        p_values[j] = max(p_values[j], 1/n_bootstrap)
    
    return coefs_mean, coefs_std, p_values, feature_names, optimal_alpha


## 6. Forest Plots

In [ ]:
def create_forest_plot(task_name, coefficients, std_errors, p_values_fdr, features, 
                        significant, top_n=20, output_dir=None):
    """Create a forest plot showing top features with confidence intervals"""
    
    df = pd.DataFrame({
        'feature': features,
        'coef': coefficients,
        'std': std_errors,
        'p_fdr': p_values_fdr,
        'sig': significant
    })
    
    df['abs_coef'] = np.abs(df['coef'])
    df = df.sort_values('abs_coef', ascending=False).head(top_n)
    df = df.sort_values('coef')
    
    fig, ax = plt.subplots(figsize=(10, max(8, top_n * 0.4)))
    
    colors = ['#E74C3C' if sig else '#95A5A6' for sig in df['sig']]
    
    y_pos = np.arange(len(df))
    
    # Plot error bars individually to avoid RGBA issue
    for idx, (coef_val, std_val, color) in enumerate(zip(df['coef'], df['std'], colors)):
        ax.errorbar(coef_val, y_pos[idx], xerr=1.96*std_val,
                    fmt='o', markersize=8, capsize=5, capthick=2,
                    color='none', ecolor=color, elinewidth=2)
    
    ax.scatter(df['coef'], y_pos, c=colors, s=100, zorder=3, edgecolors='black', linewidths=1)
    
    ax.axvline(x=0, color='black', linestyle='--', linewidth=1.5, alpha=0.5)
    
    display_names = []
    for feat in df['feature']:
        name = feat.replace('_mean', '').replace('_median', '').replace('_std', '')
        name = name.replace('qbc_', '').replace('num_', '')
        if len(name) > 30:
            name = name[:27] + '...'
        display_names.append(name)
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels(display_names, fontsize=9)
    ax.set_xlabel('Standardized Coefficient', fontsize=12, fontweight='bold')
    ax.set_title(f'{task_name}\nTop {top_n} Features Predicting Quantum Advantage', 
                fontsize=13, fontweight='bold', pad=20)
    
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#E74C3C', label='Significant (FDR < 0.05)'),
        Patch(facecolor='#95A5A6', label='Not significant')
    ]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=10)
    
    ax.grid(True, axis='x', alpha=0.3, linestyle='--')
    ax.set_axisbelow(True)
    
    plt.tight_layout()
    
    if output_dir:
        safe_name = task_name.lower().replace(' ', '_')
        plt.savefig(output_dir / f'forest_plot_{safe_name}.png', dpi=300, bbox_inches='tight')
        print(f"  Saved: forest_plot_{safe_name}.png")
    
    plt.show()
    return fig


In [ ]:
# Create forest plots
print("\n=== CREATING FOREST PLOTS ===")
for task_name, res in results.items():
    print(f"\n{task_name}:")
    create_forest_plot(
        task_name,
        res['coefficients'],
        res['std'],
        res['p_values_fdr'],
        res['features'],
        res['significant_fdr'],
        top_n=20,
        output_dir=output_dir
    )


## 7. Summary Statistics

In [ ]:
print("\n=== SUMMARY STATISTICS ===\n")

for task_name, res in results.items():
    print(f"\n{task_name}:")
    print(f"  Ridge alpha: {res['alpha']:.4f}")
    print(f"  Total features: {len(res['features'])}")
    print(f"  Significant (FDR < 0.05): {res['significant_fdr'].sum()}")
    
    sig_mask = res['significant_fdr']
    if sig_mask.sum() > 0:
        sig_features = np.array(res['features'])[sig_mask]
        sig_coefs = res['coefficients'][sig_mask]
        
        print(f"\n  Significant features:")
        for feat, coef in sorted(zip(sig_features, sig_coefs), key=lambda x: abs(x[1]), reverse=True):
            direction = "↑" if coef > 0 else "↓"
            print(f"    {direction} {feat}: {coef:.4f}")
    else:
        print("  No features reached FDR significance threshold")

print("\n" + "="*60)
print("ANALYSIS COMPLETE!")
print("="*60)
print(f"\nGenerated files in {output_dir}:")
print("  - ridge_coefficients_ranking.csv")
print("  - ridge_coefficients_classification.csv")
print("  - ridge_coefficients_link_prediction.csv")
print("  - forest_plot_ranking.png")
print("  - forest_plot_classification.png")
print("  - forest_plot_link_prediction.png")

## Summary

Identified complexity features predicting quantum vs classical performance advantages.

### Methodology:
1. **Data Cleaning**: Removed columns with >50% NaNs
2. **Ridge Regression**: Cross-validated L2 regularization
3. **Bootstrapping**: 1000 samples for coefficient distributions
4. **P-values**: Two-tailed test (H0: coefficient = 0)
5. **FDR Correction**: Benjamini-Hochberg (α = 0.05)

### Statistical Rigor:
- Standardized features (mean=0, std=1)
- 5-fold cross-validation for alpha
- 95% bootstrap confidence intervals
- Multiple testing correction (FDR < 0.05)
- Supports both statsmodels and custom FDR

### Interpretation:
- **Positive coefficients**: Features favoring quantum methods
- **Negative coefficients**: Features favoring classical methods
- **Significance**: FDR < 0.05 threshold

### Output:
- CSV files with all coefficients and p-values
- Forest plots (top 20 features per task)
- Red = significant, Gray = not significant